# The Central Limit Theorem
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what the Central Limit Theorem states in plain English
- **Describe** how the sampling distribution of the mean becomes more normal as sample size increases
- **Interpret** why a larger sample size gives a narrower sampling distribution

> **Tip:** Start with the **sample size slider** at n = 2, the distribution of sample means will look nothing like a bell curve. Then drag it to n = 30 and watch the transformation.

---
## How we got here

In *12: The Normal Distribution* we saw that the normal distribution describes heights, test scores, and measurement error. But why does the bell curve show up in so many unrelated places? The Central Limit Theorem (CLT) is the answer, it proves that any population's sample means will form a normal distribution, regardless of the population's own shape, as long as the sample is large enough.

---
## Why this matters for data science

The CLT is the theorem that makes statistical inference possible. Every time you compute a confidence interval or run a hypothesis test on sample data, you are relying on the CLT to guarantee that your sample statistic follows a predictable distribution. Without it, statistics as a discipline would not exist in its current form.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots
from ipywidgets import HTML

# ── Population definitions ────────────────────────────────────────────────────
def _bimodal_sample(n):
    mask = np.random.rand(n) < 0.5
    return np.where(mask, np.random.normal(-2, 0.7, n), np.random.normal(2, 0.7, n))

_BIMODAL_SIGMA = np.sqrt(0.5 * (0.7**2 + 4) + 0.5 * (0.7**2 + 4))

POPULATIONS = {
    "Uniform": {
        "mu": 0.5,  "sigma": 1 / np.sqrt(12),
        "sample": lambda n: np.random.uniform(0, 1, n),
        "pdf_x":  np.linspace(-0.05, 1.05, 300),
        "pdf_y":  lambda x: np.where((x >= 0) & (x <= 1), 1.0, 0.0),
        "xrange": [-0.15, 1.15],
        "display": "Uniform(0, 1)",
    },
    "Exponential": {
        "mu": 1.0,  "sigma": 1.0,
        "sample": lambda n: np.random.exponential(1.0, n),
        "pdf_x":  np.linspace(0.0, 6.0, 300),
        "pdf_y":  lambda x: stats.expon.pdf(x, scale=1.0),
        "xrange": [-0.2, 6.0],
        "display": "Exponential(λ=1)",
    },
    "Bimodal": {
        "mu": 0.0,  "sigma": _BIMODAL_SIGMA,
        "sample": _bimodal_sample,
        "pdf_x":  np.linspace(-5.0, 5.0, 300),
        "pdf_y":  lambda x: (
            0.5 * stats.norm.pdf(x, -2, 0.7) + 0.5 * stats.norm.pdf(x, 2, 0.7)
        ),
        "xrange": [-5.0, 5.0],
        "display": "Bimodal (peaks at ±2)",
    },
    "Right-Skewed": {
        "mu": 2.0,  "sigma": np.sqrt(2.0),
        "sample": lambda n: np.random.gamma(2, 1, n),
        "pdf_x":  np.linspace(0.0, 8.0, 300),
        "pdf_y":  lambda x: stats.gamma.pdf(x, a=2, scale=1),
        "xrange": [-0.2, 8.0],
        "display": "Right-Skewed (Gamma, k=2)",
    },
}

def _draw_means(pop_name, n, num_samples):
    pop = POPULATIONS[pop_name]
    if pop_name == "Bimodal":
        mask = np.random.rand(num_samples, n) < 0.5
        a = np.random.normal(-2, 0.7, (num_samples, n))
        b = np.random.normal(2, 0.7, (num_samples, n))
        return np.where(mask, a, b).mean(axis=1)
    return pop["sample"](num_samples * n).reshape(num_samples, n).mean(axis=1)

# ── State ─────────────────────────────────────────────────────────────────────
_state = {"means": [], "last_sample": None}

# ── Persistent FigureWidget — built once, updated in-place ────────────────────
# Using a persistent FigureWidget (not Output + display) avoids duplicate
# rendering: widgets placed directly in a VBox never leak a second copy.
_sub = make_subplots(
    rows=1, cols=3,
    subplot_titles=["① Population", "② One Sample", "③ Sampling Distribution"],
    column_widths=[0.27, 0.31, 0.42],
    horizontal_spacing=0.10,
)
# T0: Population PDF
_sub.add_trace(go.Scatter(x=[], y=[], fill="tozeroy",
    line=dict(color=PALETTE["primary"], width=2.5),
    opacity=0.75, showlegend=False), row=1, col=1)
# T1: Sample dots
_sub.add_trace(go.Scatter(x=[], y=[], mode="markers", visible=False,
    marker=dict(color=PALETTE["secondary"], size=9, opacity=0.72,
                line=dict(color="white", width=1)),
    showlegend=False), row=1, col=2)
# T2: Sample mean line (2-point vertical line trace)
_sub.add_trace(go.Scatter(x=[], y=[], mode="lines", visible=False,
    line=dict(color=PALETTE["accent"], width=3),
    showlegend=False), row=1, col=2)
# T3: Sampling distribution histogram
_sub.add_trace(go.Histogram(x=[0.0], histnorm="probability density",
    nbinsx=5, visible=False,
    marker_color=PALETTE["secondary"], opacity=0.7, showlegend=False), row=1, col=3)
# T4: CLT normal overlay
_sub.add_trace(go.Scatter(x=[], y=[], mode="lines", visible=False,
    line=dict(color=PALETTE["primary"], width=2.5, dash="dash"),
    showlegend=False), row=1, col=3)
# T5: Last mean highlight (dotted line in panel 3)
_sub.add_trace(go.Scatter(x=[], y=[], mode="lines", visible=False,
    line=dict(color=PALETTE["accent"], width=2.5, dash="dot"),
    showlegend=False), row=1, col=3)

_bl = base_layout(
    title="Central Limit Theorem — from a single draw to the bell curve",
    xaxis_title="", yaxis_title="",
)
_bl.update(height=430, showlegend=False, margin=dict(t=100, b=50, l=50, r=30))
_sub.update_layout(_bl)
_sub.update_xaxes(title_text="Value",            row=1, col=1)
_sub.update_yaxes(title_text="Density",           row=1, col=1)
_sub.update_xaxes(title_text="Value (one draw)",  row=1, col=2)
_sub.update_yaxes(visible=False, range=[-0.5, 0.5], row=1, col=2)
_sub.update_xaxes(title_text="Sample Mean (X̄)",  row=1, col=3)
_sub.update_yaxes(title_text="Density",           row=1, col=3)

fig_widget = go.FigureWidget(_sub)
_n_fixed_anns = len(fig_widget.layout.annotations)   # 3 subplot title annotations

# ── Controls ──────────────────────────────────────────────────────────────────
pop_dd = Dropdown(
    options=list(POPULATIONS.keys()), value="Uniform",
    description="Population:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="380px"),
)
n_sl = IntSlider(
    value=5, min=1, max=100, step=1,
    description="Sample size (n):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="460px"),
    continuous_update=False,
)
nsamp_sl = IntSlider(
    value=500, min=100, max=5000, step=100,
    description="Batch size:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="460px"),
    continuous_update=False,
)
draw_one_btn = Button(
    description="Draw 1 sample", button_style="primary", icon="arrow-right",
    layout=widgets.Layout(width="175px", height="36px"),
)
draw_many_btn = Button(
    description="Draw batch", icon="forward",
    layout=widgets.Layout(width="140px", height="36px"),
)
reset_btn = Button(
    description="Reset", button_style="warning", icon="undo",
    layout=widgets.Layout(width="100px", height="36px"),
)
summary_html = HTML(
    value=(
        f'<p style="font-family:{FONT["family"]};color:#888;margin:6px 0;">'
        f'Click <b>Draw 1 sample</b> to step through the process one draw at a time, '
        f'or <b>Draw batch</b> to fill in the sampling distribution quickly.</p>'
    )
)

# ── Render ─────────────────────────────────────────────────────────────────────
def render():
    pop_name  = pop_dd.value
    n         = n_sl.value
    means     = _state["means"]
    last_samp = _state["last_sample"]
    pop       = POPULATIONS[pop_name]
    mu, sigma = pop["mu"], pop["sigma"]
    se        = sigma / np.sqrt(n)
    num_means = len(means)

    with fig_widget.batch_update():
        # T0: Population PDF
        fig_widget.data[0].x = pop["pdf_x"]
        fig_widget.data[0].y = pop["pdf_y"](pop["pdf_x"])

        # T1 & T2: Sample dots + mean line
        if last_samp is not None:
            jitter = np.random.default_rng(42).uniform(-0.28, 0.28, len(last_samp))
            smean  = float(last_samp.mean())
            fig_widget.data[1].x       = last_samp
            fig_widget.data[1].y       = jitter
            fig_widget.data[1].visible = True
            fig_widget.data[2].x       = [smean, smean]
            fig_widget.data[2].y       = [-0.42, 0.42]
            fig_widget.data[2].visible = True
        else:
            fig_widget.data[1].visible = False
            fig_widget.data[2].visible = False

        # T3, T4, T5: Sampling distribution
        if means:
            means_arr = np.array(means)
            fig_widget.data[3].x      = means_arr
            fig_widget.data[3].nbinsx = min(40, max(5, num_means // 10))
            fig_widget.data[3].visible = True
            x_clt = np.linspace(mu - 4.5 * se, mu + 4.5 * se, 300)
            fig_widget.data[4].x       = x_clt
            fig_widget.data[4].y       = stats.norm.pdf(x_clt, mu, se)
            fig_widget.data[4].visible = True
            if last_samp is not None:
                smean  = float(last_samp.mean())
                peak_y = stats.norm.pdf(mu, mu, se)
                fig_widget.data[5].x       = [smean, smean]
                fig_widget.data[5].y       = [0.0, peak_y * 1.35]
                fig_widget.data[5].visible = True
            else:
                fig_widget.data[5].visible = False
        else:
            fig_widget.data[3].visible = False
            fig_widget.data[4].visible = False
            fig_widget.data[5].visible = False

        # Axis ranges (panel 1 & 2 follow the selected population)
        fig_widget.layout.xaxis.range  = pop["xrange"]
        fig_widget.layout.xaxis2.range = pop["xrange"]

        # Annotations: preserve the 3 fixed subplot titles, then add custom ones
        anns = list(fig_widget.layout.annotations)[:_n_fixed_anns]

        if last_samp is not None:
            smean = float(last_samp.mean())
            anns.append(dict(
                text=f"<b>x̄ = {smean:.3f}</b>",
                x=smean, y=0.40, xref="x2", yref="y2",
                showarrow=False, yanchor="top",
                font=dict(size=12, family=FONT["family"], color=PALETTE["accent"]),
                bgcolor="rgba(255,255,255,0.88)",
                bordercolor=PALETTE["accent"], borderwidth=1, borderpad=3,
            ))
        else:
            anns.append(dict(
                text='Click "Draw 1 sample" to see a single draw',
                xref="x2 domain", yref="y2 domain",
                x=0.5, y=0.5, xanchor="center", yanchor="middle",
                showarrow=False,
                font=dict(size=12, family=FONT["family"], color="#aaa"),
            ))

        if not means:
            anns.append(dict(
                text="Each draw adds one mean here — do it enough times → bell curve.",
                xref="x3 domain", yref="y3 domain",
                x=0.5, y=0.5, xanchor="center", yanchor="middle",
                showarrow=False,
                font=dict(size=12, family=FONT["family"], color="#aaa"),
            ))

        fig_widget.layout.annotations = anns

    # Summary panel (separate widget, updated outside batch_update)
    if means:
        means_arr = np.array(means)
        emp_mean  = float(np.mean(means_arr))
        emp_std   = float(np.std(means_arr))
        if n < 5:
            note, nc = "n is very small — shape still mirrors the population", "#c0392b"
        elif n < 30:
            note, nc = "Bell shape emerging", "#e67e22"
        else:
            note, nc = "CLT in full effect — approximately normal regardless of population shape", "#27ae60"
        summary_html.value = (
            f'<div style="font-family:{FONT["family"]}; background:#f8f9fb; '
            f'border-left:4px solid {PALETTE["primary"]}; border-radius:6px; '
            f'padding:12px 18px; margin-top:4px; font-size:13px; line-height:1.8;">'
            f'<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px;">'
            f'<div><b>Population</b><br>{pop["display"]}<br>μ={mu:.4f}, σ={sigma:.4f}</div>'
            f'<div><b>Empirical (X̄ values)</b><br>Mean of means: {emp_mean:.4f}<br>'
            f'Std of means: {emp_std:.4f}</div>'
            f'<div><b>CLT Prediction</b><br>μ_x̄ = μ = {mu:.4f}<br>'
            f'σ/√n = {sigma:.4f}/√{n} = {se:.4f}</div>'
            f'</div>'
            f'<div style="margin-top:10px;padding:6px 10px;background:{nc}1a;'
            f'border-radius:4px;color:{nc};font-weight:600;font-size:12.5px;">'
            f'n = {n},  {num_means:,} samples drawn:  {note}</div>'
            f'</div>'
        )

# ── Event handlers ─────────────────────────────────────────────────────────────
def _on_draw_one(b):
    sample = POPULATIONS[pop_dd.value]["sample"](n_sl.value)
    _state["last_sample"] = sample
    _state["means"].append(float(np.mean(sample)))
    render()

def _on_draw_many(b):
    pop_name = pop_dd.value
    n        = n_sl.value
    new_means = _draw_means(pop_name, n, nsamp_sl.value)
    _state["last_sample"] = POPULATIONS[pop_name]["sample"](n)
    _state["means"].extend(new_means.tolist())
    render()

def _on_reset(b):
    _state["means"] = []
    _state["last_sample"] = None
    render()

def _on_param_change(change):
    _state["means"] = []
    _state["last_sample"] = None
    render()

draw_one_btn.on_click(_on_draw_one)
draw_many_btn.on_click(_on_draw_many)
reset_btn.on_click(_on_reset)
pop_dd.observe(_on_param_change, names="value")
n_sl.observe(_on_param_change,   names="value")

# ── Display — fig_widget lives directly in the VBox, no Output() wrapper ──────
btn_row  = HBox([draw_one_btn, draw_many_btn, reset_btn],
                layout=widgets.Layout(gap="8px", margin="4px 0"))
controls = VBox([pop_dd, n_sl, nsamp_sl, btn_row])
display(VBox([controls, fig_widget, summary_html]))
render()


---
## What's happening?

The Central Limit Theorem states: if you draw many samples of size n from **any** population with mean μ and finite variance σ², the distribution of sample means will approach a normal distribution as n grows.

| Quantity | Symbol | Formula |
|---------|--------|---------|
| Population mean | μ | Fixed parameter of the population |
| Standard error | SE | σ / √n: standard deviation of the sampling distribution |
| Sampling distribution | X̄ ~ N(μ, SE²) | What sample means look like across many samples |

$$\bar{X} \xrightarrow{d} N\!\left(\mu, \frac{\sigma^2}{n}\right) \quad \text{as } n \to \infty$$

### Two things happen as n increases

1. The sampling distribution becomes **more normal** in shape, regardless of how weird the population looks
2. The standard error (SE = σ/√n) gets **smaller**, sample means cluster more tightly around μ

Return to the widget and verify both effects: set n = 1 (no averaging, you see the raw population), then increase to n = 5, 10, 30 and watch the distribution narrow and normalize.

---
## Real-world example: Income data becoming normal through averaging

Individual incomes are wildly right-skewed, a small number of very high earners pull the mean far above the median. But if we repeatedly draw samples of n households and compute each sample's mean income, those sample means form a nearly normal distribution.

The chart below draws 2,000 samples of size n = 35 from a right-skewed income distribution and plots the distribution of sample means alongside the original population. Notice:

- **Notice:** The population (top) is strongly right-skewed, with a long tail toward high incomes
- **Notice:** The sampling distribution of means (bottom) is approximately bell-shaped and symmetric, centered at the population mean
- **Notice:** The spread of the sampling distribution is much narrower than the population, SE = σ/√n shrinks it

> **Discussion question:** A pollster surveys 35 households to estimate average income in a city. If the true population mean is $68,000 and σ is $40,000, what is the standard error of the pollster's estimate? Within what range would 95% of such surveys land?

In [3]:
np.random.seed(99)

# ── Skewed income population and CLT demonstration ────────────────────────────
# Lognormal income population (strongly right-skewed)
pop_size = 100_000
incomes  = np.random.lognormal(mean=11.1, sigma=0.7, size=pop_size)  # ~$66k median

n_samples = 2000
n_per     = 35
sample_means = [np.mean(np.random.choice(incomes, n_per, replace=True))
                for _ in range(n_samples)]
sample_means = np.array(sample_means)

pop_mu = np.mean(incomes)
pop_se = np.std(incomes) / np.sqrt(n_per)

from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=["Population Distribution (right-skewed income)",
                                    f"Sampling Distribution of x̄  (n={n_per}, {n_samples} samples)"])

# Population histogram (clipped for readability)
fig.add_trace(go.Histogram(
    x=incomes / 1000, nbinsx=80, histnorm="probability density",
    marker_color=PALETTE["muted"], opacity=0.7, name="Population",
), row=1, col=1)

# Sampling distribution
x_norm = np.linspace(sample_means.min(), sample_means.max(), 300)
y_norm = stats.norm.pdf(x_norm, pop_mu, pop_se)
fig.add_trace(go.Histogram(
    x=sample_means / 1000, nbinsx=50, histnorm="probability density",
    marker_color=PALETTE["primary"], opacity=0.7, name="Sample means",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=x_norm / 1000, y=y_norm,
    mode="lines", line=dict(color=PALETTE["secondary"], width=2.5),
    name=f"N(μ={pop_mu/1000:.0f}k, SE={pop_se/1000:.1f}k)",
), row=2, col=1)

fig.update_xaxes(range=[0, 300], row=1, col=1, title_text="Income ($000s)")
fig.update_xaxes(range=[40, 100], row=2, col=1, title_text="Sample Mean Income ($000s)")
fig.update_layout(
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    font=dict(family=PALETTE.get("family", "Inter, Arial, sans-serif")),
    title=dict(text="The Central Limit Theorem in Action",
               font=dict(size=FONT["size_title"])),
    height=550,
)
fig.show()

### The CLT in practice: how large must n be?

| Population shape | n needed for approximate normality |
|-----------------|-----------------------------------|
| Already normal | n = 1 (trivially satisfied) |
| Symmetric, slightly non-normal | n ≈ 10–15 |
| Moderately skewed | n ≈ 30 (the common "rule of thumb") |
| Heavily skewed or heavy-tailed | n ≈ 100+ |
| Extremely skewed (e.g., power law) | May need n = 1000+ |

---
## Key takeaway

> **The Central Limit Theorem guarantees that sample means become normally distributed as n grows, regardless of population shape — this is the mathematical bedrock of all statistical inference.**

---
*Next up: Sampling Distributions — building on the CLT to describe exactly how any sample statistic varies across repeated samples*